<a href="https://colab.research.google.com/github/ljvillarrealm/SMAZ-2026/blob/main/HW_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import io
import time

import numpy as np
import pandas as pd
import requests
import yfinance as yf

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/140.0 Safari/537.36"}

def read_html_table(url):
    resp = requests.get(url, headers=HEADERS, timeout=60)
    resp.raise_for_status()
    return pd.read_html(io.StringIO(resp.text))[0]

def to_number(series):
    return pd.to_numeric(series.astype(str).str.replace(r"[$,%]", "", regex=True), errors="coerce")

## Question 1: Withdrawn IPOs by Company Type

In [2]:
filed = read_html_table("https://www.iposcoop.com/ipos-recently-filed/")
filed["File Date"] = pd.to_datetime(filed["File Date"])
print("rows in table:", len(filed), "| file dates:", filed["File Date"].min().date(), "->", filed["File Date"].max().date())

withdrawn = filed[(filed["Expected To Trade"] == "Withdrawn")
                  & (filed["File Date"] < "2026-09-11")].copy()
print("withdrawn before 2026-09-11:", len(withdrawn))

rows in table: 500 | file dates: 2025-12-23 -> 2026-09-24
withdrawn before 2026-09-11: 31


In [3]:
# Ordered rules: first match wins (plain substring matching, so "Technology" != "Technologies")
RULES = [
    ("Technologies", ["Technologies"]),
    ("Acquisition Corp", ["Acquisition Corp", "Acquisition Corporation", "Corp"]),
    ("Inc.", ["Inc", "Incorporated"]),
    ("Group", ["Group"]),
    ("Limited", ["Ltd", "Limited"]),
    ("Holdings", ["Holdings", "Holding"]),
]

def company_type(name):
    for label, patterns in RULES:
        if any(p in name for p in patterns):
            return label
    return "Other"

def parse_price(value):
    if pd.isna(value) or str(value).strip() in ("", "-"):
        return None
    return float(str(value).replace("$", "").replace(",", ""))

withdrawn["Company Type"] = withdrawn["Company"].apply(company_type)
withdrawn["Avg_price"] = (withdrawn["Price Low"].apply(parse_price).astype(float)
                          + withdrawn["Price High"].apply(parse_price).astype(float)) / 2
withdrawn["Shares (millions)"] = to_number(withdrawn["Shares (millions)"])
withdrawn["Est $ Vol (millions)"] = to_number(withdrawn["Est $ Vol (millions)"])
withdrawn["Shares_offered_value"] = (withdrawn["Shares (millions)"] * withdrawn["Avg_price"]).fillna(
    withdrawn["Est $ Vol (millions)"])

withdrawn[["Company", "Company Type", "Shares (millions)", "Avg_price",
           "Est $ Vol (millions)", "Shares_offered_value"]]

,Company,Company Type,Shares (millions),Avg_price,Est $ Vol (millions),Shares_offered_value
38,Motive Technologies (Withdrawn),Technologies,0.00,NaN,100.00,100.0000
43,Idea Tech Holding (Withdrawn),Holdings,2.00,4.50,9.00,9.0000
50,Hornbeck Offshore Services (Withdrawn),Other,0.00,NaN,100.00,100.0000
52,Coolbit Technologies Ltd. (Withdrawn),Technologies,5.00,4.50,22.50,22.5000
59,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp,20.00,10.00,200.00,200.0000
69,pan-Africa Corp. (Withdrawn),Acquisition Corp,12.50,10.00,125.00,125.0000
84,Living Homeopathy (Withdrawn),Other,3.75,5.00,19.00,18.7500
128,Nuclea Energy (Withdrawn),Other,5.60,9.00,50.40,50.4000
158,APEX Global Solutions Ltd. (WITHDRAWN),Limited,3.75,4.00,15.00,15.0000
163,Cloud Data Holdings (Withdrawn),Holdings,3.75,4.25,15.94,15.9375


In [4]:
q1 = withdrawn.groupby("Company Type")["Shares_offered_value"].agg(["count", "sum"]).sort_values("sum", ascending=False)
q1

,count,sum
Company Type,,
Acquisition Corp,5,499.9850
Inc.,1,351.0000
Holdings,4,311.6575
Other,8,290.4450
Limited,7,197.2500
Technologies,3,184.9000
Group,3,32.5000


499.9985 ~ 500

## Question 2: Median Sharpe Ratio for 2025 IPOs (first 8 months)

In [5]:
ipos_2025 = read_html_table("https://www.iposcoop.com/2025-pricings/")
ipos_2025["Offer Date"] = pd.to_datetime(ipos_2025["Offer Date"], format="%m/%d/%Y")
ipos_2025["Return"] = to_number(ipos_2025["Return"])
print("2025 IPOs:", len(ipos_2025))

ipos_filtered = ipos_2025[(ipos_2025["Offer Date"] < "2025-09-01") & (ipos_2025["Return"] != 0)]
tickers = ipos_filtered["Symbol"].unique().tolist()
print("filtered tickers:", len(tickers))

2025 IPOs: 231
filtered tickers: 146


In [ ]:
frames, missing = [], []
for ticker in tickers:
    try:
        df = yf.download(ticker, start="2025-01-01", end="2026-09-12",
                         progress=False, auto_adjust=True)
    except Exception as exc:
        print(ticker, "error:", exc)
        df = pd.DataFrame()
    if df.empty:
        missing.append(ticker)
        continue
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.sort_index()
    df["Ticker"] = ticker
    # features are computed per ticker so rolling windows never cross stocks
    df["growth_252d"] = df["Close"] / df["Close"].shift(252)
    df["volatility"] = df["Close"].rolling(30).std() * np.sqrt(252)
    df["Sharpe"] = (df["growth_252d"] - 0.05) / df["volatility"]
    for m in range(1, 13):
        df[f"future_growth_{m}_m"] = df["Close"].shift(-21 * m) / df["Close"]
    frames.append(df)
    time.sleep(0.2)

stocks_df = pd.concat(frames)
stocks_df.index.name = "Date"
print("downloaded:", stocks_df["Ticker"].nunique(), "| missing:", len(missing), missing)

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MJID']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EMPG"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['EMPG']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['CAEP']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['PTNM']: YFTzMissingError('possibly delisted; no timezone found')


In [ ]:
q2 = stocks_df.loc[stocks_df.index == "2026-09-11", ["Ticker", "growth_252d", "volatility", "Sharpe"]]
q2.describe()

In [ ]:
print("median Sharpe on 2026-09-11:", round(q2["Sharpe"].median(), 4))
print("growth_252d median vs mean:", round(q2["growth_252d"].median(), 4), round(q2["growth_252d"].mean(), 4))
print("stocks with 252 trading days:", q2["growth_252d"].notna().sum())
q2.sort_values("Sharpe", ascending=False).head(10)

nearest value to the calculated media is 0.04

## Question 3: Fixed Months Holding Strategy

In [ ]:
growth_cols = [f"future_growth_{m}_m" for m in range(1, 13)]
growth_df = stocks_df.reset_index()[["Date", "Ticker", "Close"] + growth_cols]

min_dates = growth_df.groupby("Ticker")["Date"].min().reset_index().rename(columns={"Date": "min_date"})
first_day = min_dates.merge(growth_df, left_on=["Ticker", "min_date"], right_on=["Ticker", "Date"], how="inner")
print("tickers with entry point:", len(first_day))

q3 = first_day[growth_cols].describe()
q3

In [ ]:
medians = q3.loc["50%"]
print("best holding period:", medians.idxmax(), "-> median growth", round(medians.max(), 4))
pd.DataFrame({"median": medians, "mean": q3.loc["mean"]})

0.9354 months seems to be the best time for holding, but most near is 1.

## Question 4: Simple RSI-Based Trading Strategy

In [ ]:
import gdown

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=True)
df = pd.read_parquet("data.parquet", engine="pyarrow")
print(df.shape)
[c for c in df.columns if "rsi" in c.lower() or c in ("Date", "Ticker", "growth_future_30d")]

In [ ]:
RSI_THRESHOLD = 30
selected_df = df[(df["rsi"] < RSI_THRESHOLD)
                 & (df["Date"] >= "2000-01-01") & (df["Date"] <= "2025-06-01")]

net_income = 1000 * (selected_df["growth_future_30d"] - 1).sum()
print("trades:", len(selected_df))
print("avg 30d return: {:.2%}".format(selected_df["growth_future_30d"].mean() - 1))
print("win rate: {:.2%}".format((selected_df["growth_future_30d"] > 1).mean()))
print("net income ($K): {:.1f}".format(net_income / 1000))

net income stans for 65.8, so nearest option is 65.


## Q5 (Exploratory): Predicting a Positive-Return IPO

Ideas to improve on a naive "buy every IPO at the first close and hold" strategy:

1. **Don't buy on day one.** The first-day pop goes to allocation holders, so buying at the first close is structurally expensive. Wait for the post-IPO drift, the quiet-period expiry (~25 days) or the lock-up expiry (~90-180 days), when insider selling pressure is already priced in, and enter then.
2. **Filter the universe.** Drop SPACs/blank-check units and micro-cap, tiny-float deals (sub-$5 offer, under $50M raised) that dominate the negative tail. Prefer top-tier underwriters, deals priced above range or upsized (a demand signal), and profitable or fast-growing fundamentals.
3. **Use momentum/technical confirmation.** Enter only when the stock trades above its offer price and its 20/50-day moving averages, or on RSI-oversold rebounds, instead of on a fixed calendar date.
4. **Condition on market regime.** IPO returns are very cyclical. Gate entries by the S&P 500/Nasdaq trend, the VIX level and the trend of IPO ETFs (e.g. IPO, FPX).
5. **Manage the exit.** Use stop-losses and trailing stops, take profit at a target, and cap the holding time, since the median growth decays with the holding period.
6. **Model it.** Train a classifier (e.g. gradient boosting) predicting P(3-6 month return > 0) from deal features (sector, size, price vs. range, underwriters, float, first-day return) plus early trading features. Trade only the top-probability bucket, and size positions by confidence and volatility (Sharpe-aware) instead of a flat $1000.